# 03 - Model Interpretability

## Objective

This notebook explains the credit risk models using feature importance, SHAP and LIME.

The goal is to understand both global model behavior and local individual predictions.

In [ ]:
import sys
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

import shap
from lime.lime_tabular import LimeTabularExplainer

warnings.filterwarnings("ignore")

sys.path.append("..")

In [ ]:
DATA_PATH = "../data/echantillon_etu_tr_2.csv"

df = pd.read_csv(DATA_PATH)

df_model = df.copy()

if "ID" in df_model.columns:
    df_model = df_model.drop(columns=["ID"])

if "tx_imp" in df_model.columns:
    df_model = df_model.drop(columns=["tx_imp"])

if "ancienneté1" in df_model.columns:
    df_model["ancienneté1"] = df_model["ancienneté1"].fillna(df_model["ancienneté1"].mean())

if "engagement2" in df_model.columns:
    df_model["engagement2"] = df_model["engagement2"].fillna(df_model["engagement2"].mean())

if "type_imp" in df_model.columns:
    df_model["type_imp"] = df_model["type_imp"].fillna("No incident")

X = df_model.drop("défaut", axis=1)
y = df_model["défaut"]

X_encoded = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb_model.fit(X_train, y_train)

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": gb_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

feature_importance

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(feature_importance["Feature"], feature_importance["Importance"])
plt.gca().invert_yaxis()
plt.title("Feature Importance")
plt.xlabel("Importance")
plt.grid(True)
plt.show()

In [ ]:
explainer = shap.TreeExplainer(gb_model)

shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test)

In [ ]:
shap.summary_plot(shap_values, X_test, plot_type="bar")

In [ ]:
X_train_lime = X_train.values
X_test_lime = X_test.values

feature_names = X_train.columns.tolist()

lime_explainer = LimeTabularExplainer(
    training_data=X_train_lime,
    feature_names=feature_names,
    class_names=["No Default", "Default"],
    mode="classification",
    discretize_continuous=True,
    random_state=42
)

In [ ]:
i = 0

exp = lime_explainer.explain_instance(
    data_row=X_test_lime[i],
    predict_fn=gb_model.predict_proba,
    num_features=min(10, X_train_lime.shape[1])
)

exp.show_in_notebook(show_table=True)

## Interpretability conclusion

Feature importance, SHAP and LIME help explain the model from different perspectives:

- Feature importance identifies the most influential variables globally.
- SHAP provides a global and local view of variable contributions.
- LIME explains a specific individual prediction.

These methods are important in credit risk modeling because banking models require transparency and explainability.